Merge all datasets into NORMAL and PNEUMONIA folders
Re-split 70/15/15 stratified by class

In [1]:
import os
import shutil

from sklearn.model_selection import train_test_split

In [2]:
BASE_DIR = '../data/raw'
SPLITS = ['train', 'test', 'val']
CLASSES = ['NORMAL', 'PNEUMONIA']

In [3]:
def get_unique_path(path):
    base, ext = os.path.splitext(path)
    counter = 1
    while os.path.exists(path):
        path = f"{base}_{counter}{ext}"
        counter += 1
    return path

os.makedirs('../data/raw/merged/NORMAL', exist_ok=True)
os.makedirs('../data/raw/merged/PNEUMONIA', exist_ok=True)

In [4]:
for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(BASE_DIR, split, cls)
        dst_dir = os.path.join(BASE_DIR, 'merged', cls)

        os.makedirs(dst_dir, exist_ok=True)

        if not os.path.exists(src_dir):
            continue

        for file in os.listdir(src_dir):
            if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                continue

            src_path = os.path.join(src_dir, file)
            dst_path = os.path.join(dst_dir, file)

            dst_path = get_unique_path(dst_path)

            shutil.copy2(src_path, dst_path)

In [5]:
OUTPUT_DIR = '../data/processed'
SOURCE_DIR = os.path.join(BASE_DIR, 'merged')

SPLITS = {
    'train': 0.70,
    'val': 0.15,
    'test': 0.15
}

RANDOM_SEED = 42

In [6]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
files = []
labels = []

In [8]:
for cls in os.listdir(SOURCE_DIR):
    cls_dir = os.path.join(SOURCE_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            files.append(os.path.join(cls_dir, f))
            labels.append(cls)

print(f"Total files: {len(files)}")

Total files: 5856


In [9]:
train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    files, labels, test_size=SPLITS['test'], stratify=labels, random_state=RANDOM_SEED
)

val_ratio_adjusted = SPLITS['val'] / (SPLITS['train'] + SPLITS['val'])  # adjust ratio in remaining data
train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files, train_val_labels, test_size=val_ratio_adjusted,
    stratify=train_val_labels, random_state=RANDOM_SEED
)

def move_files(file_list, label_list, split_name):
    for file_path, label in zip(file_list, label_list):
        dst_dir = os.path.join(OUTPUT_DIR, split_name, label)
        os.makedirs(dst_dir, exist_ok=True)
        dst_file = os.path.join(dst_dir, os.path.basename(file_path))
        # Ensure no overwrite
        base, ext = os.path.splitext(dst_file)
        counter = 1
        while os.path.exists(dst_file):
            dst_file = f"{base}_{counter}{ext}"
            counter += 1
        shutil.move(file_path, dst_file)

In [10]:
move_files(train_files, train_labels, 'train')
move_files(val_files, val_labels, 'val')
move_files(test_files, test_labels, 'test')

In [11]:
if os.path.exists(SOURCE_DIR):
    shutil.rmtree(SOURCE_DIR)

In [12]:
for split in ['train', 'val', 'test']:
    normal = len(os.listdir(f'../data/processed/{split}/NORMAL'))
    pneumonia = len(os.listdir(f'../data/processed/{split}/PNEUMONIA'))
    total = normal + pneumonia
    print(f"{split}: NORMAL {normal/total:.1%} | PNEUMONIA {pneumonia/total:.1%}")

train: NORMAL 27.0% | PNEUMONIA 73.0%
val: NORMAL 27.1% | PNEUMONIA 72.9%
test: NORMAL 27.1% | PNEUMONIA 72.9%
